# Notebook 07 — Stage J Strong Tabular Model Expansion

Purpose:
- expand beyond Stage I with stronger tabular learners
- compare fairly against frozen Stage I baseline
- promote only if performance and safety both improve

## Planned Tasks (Notebook 07)

- [x] Load Stage I selected model and frozen split governance
- [x] Rebuild canonical feature matrix from Stage F panel (same contract)
- [x] Train Stage J family set (RF, XGBoost, LightGBM; optional KNN/SVM diagnostics)
- [x] Generate unified H/I/J comparison matrix with AUROC, PR-AUC, Brier, ECE
- [x] Add bootstrap CI + subgroup spread deltas for promotion safety
- [x] Export recommendation report + Stage J manifest + checklist proof

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook07_stage_j'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook07_stage_j'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook07_stage_j'
META_DIR = PROJECT_ROOT / 'Data' / 'metadata'
for d in [TABLE_DIR, FIG_DIR, REPORT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Notebook 07 workspace ready')

Notebook 07 workspace ready


In [2]:
# Stage I handoff loading
phase_i_manifest_path = META_DIR / 'phase_i_manifest.json'
phase_i_proof_path = PROJECT_ROOT / 'Results' / 'reports' / 'notebook06_stage_i' / 'stage_i_checklist_proof.json'

if not phase_i_manifest_path.exists():
    raise FileNotFoundError(f'Missing Stage I manifest: {phase_i_manifest_path}')
if not phase_i_proof_path.exists():
    raise FileNotFoundError(f'Missing Stage I checklist proof: {phase_i_proof_path}')

with open(phase_i_manifest_path, 'r', encoding='utf-8') as f:
    phase_i_manifest = json.load(f)
with open(phase_i_proof_path, 'r', encoding='utf-8') as f:
    phase_i_proof = json.load(f)

selected = phase_i_proof.get('selected', {})
print('Stage I selected family:', selected.get('family', 'unknown'))
print('Stage I selected model:', selected.get('model', 'unknown'))
print('Stage I manifest outputs reports:', len(phase_i_manifest.get('outputs_reports', [])))

Stage I selected family: random_forest
Stage I selected model: rf_n320_d12_leaf5
Stage I manifest outputs reports: 4


In [5]:
# Stage J implementation block 1: canonical data assembly + strong tabular family sweep
import time
import warnings
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
np.random.seed(42)

comparison_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook04_stage_g' / 'stage_g_model_stage_comparison.csv'

panel_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_f' / 'phase_f_closed_loop_panel.parquet'
if not panel_path.exists():
    raise FileNotFoundError(f'Missing Stage F panel: {panel_path}')

cols = [
    'patient_id', 'day', 'I_stage_f_base', 'hazard_prob_stage_f', 'stage_f_escalation_event',
    'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active'
 ]
data = pd.read_parquet(panel_path, columns=cols).copy()

# deterministic messy-signal continuity from Stage G/I
p_missing_instability = np.clip(0.03 + 0.12 * (data['response_state_active'].eq('nonresponse')).astype(float), 0.0, 0.35)
p_missing_hazard = np.clip(0.02 + 0.10 * (data['months_since_cycle_start'] > 18).astype(float), 0.0, 0.25)
p_missing_cycle = np.clip(0.01 + 0.08 * (data['stage_f_escalation_event'] == 0).astype(float), 0.0, 0.20)

u1 = np.random.rand(len(data))
u2 = np.random.rand(len(data))
u3 = np.random.rand(len(data))
data.loc[u1 < p_missing_instability, 'I_stage_f_base'] = np.nan
data.loc[u2 < p_missing_hazard, 'hazard_prob_stage_f'] = np.nan
data.loc[u3 < p_missing_cycle, 'months_since_cycle_start'] = np.nan

for c in ['I_stage_f_base', 'hazard_prob_stage_f', 'months_since_cycle_start']:
    data[f'{c}__is_missing'] = data[c].isna().astype(np.int8)

data['is_nonresponse'] = data['response_state_active'].eq('nonresponse').astype(np.int8)
data['is_partial'] = data['response_state_active'].eq('partial_response').astype(np.int8)
data['is_stabilized'] = data['response_state_active'].eq('stabilized').astype(np.int8)
data['I_stage_f_base_x_cycle'] = data['I_stage_f_base'].fillna(data['I_stage_f_base'].median()) * data['cycle_id_stage_f']
data['extreme_hazard_flag'] = (
    data['hazard_prob_stage_f'].fillna(data['hazard_prob_stage_f'].median())
    >= data['hazard_prob_stage_f'].fillna(data['hazard_prob_stage_f'].median()).quantile(0.995)
).astype(np.int8)

label = 'stage_f_escalation_event'
pos = data[data[label] == 1]
neg = data[data[label] == 0].sample(n=min(len(data[data[label] == 0]), len(pos) * 3), random_state=42)
model_df = pd.concat([pos, neg], axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)

patients = model_df['patient_id'].drop_duplicates().sample(frac=1.0, random_state=42).to_numpy()
n = len(patients)
p_train = set(patients[: int(0.70 * n)])
p_val = set(patients[int(0.70 * n): int(0.85 * n)])
p_test = set(patients[int(0.85 * n):])
model_df['split'] = np.where(
    model_df['patient_id'].isin(p_train),
    'train',
    np.where(model_df['patient_id'].isin(p_val), 'val', 'test')
)

feature_set = [
    'cycle_id_stage_f', 'months_since_cycle_start', 'I_stage_f_base', 'hazard_prob_stage_f',
    'is_nonresponse', 'is_partial', 'is_stabilized',
    'I_stage_f_base__is_missing', 'hazard_prob_stage_f__is_missing', 'months_since_cycle_start__is_missing',
    'I_stage_f_base_x_cycle', 'extreme_hazard_flag'
 ]

X_train = model_df.loc[model_df['split'] == 'train', feature_set]
y_train = model_df.loc[model_df['split'] == 'train', label].astype(int).to_numpy()
X_val = model_df.loc[model_df['split'] == 'val', feature_set]
y_val = model_df.loc[model_df['split'] == 'val', label].astype(int).to_numpy()
X_test = model_df.loc[model_df['split'] == 'test', feature_set]
y_test = model_df.loc[model_df['split'] == 'test', label].astype(int).to_numpy()

def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        ece += abs(y_true[m].mean() - y_prob[m].mean()) * (m.sum() / len(y_true))
    return float(ece)

def metric_row(stage, model, split_name, y_true, y_prob, train_sec=None, pred_sec=None):
    return {
        'stage': stage,
        'model': model,
        'split': split_name,
        'n': int(len(y_true)),
        'auroc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'brier': float(brier_score_loss(y_true, y_prob)),
        'ece_10bin': ece_score(y_true, y_prob),
        'train_seconds': None if train_sec is None else float(train_sec),
        'predict_seconds': None if pred_sec is None else float(pred_sec)
    }

# Model registry (XGBoost/LightGBM optional)
model_registry = []

rf_pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), feature_set)], remainder='drop')
model_registry.append(('rf_stage_i_selected', Pipeline([
    ('pre', rf_pre),
    ('clf', RandomForestClassifier(n_estimators=320, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1))
])))

et_pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), feature_set)], remainder='drop')
model_registry.append(('extra_trees_strong', Pipeline([
    ('pre', et_pre),
    ('clf', ExtraTreesClassifier(n_estimators=420, max_depth=None, min_samples_leaf=3, random_state=42, n_jobs=-1))
])))

hgb_pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), feature_set)], remainder='drop')
model_registry.append(('hist_gb_strong', Pipeline([
    ('pre', hgb_pre),
    ('clf', HistGradientBoostingClassifier(learning_rate=0.06, max_depth=8, max_iter=350, random_state=42))
])))

knn_pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), feature_set)], remainder='drop')
model_registry.append(('knn_diag', Pipeline([
    ('pre', knn_pre),
    ('clf', KNeighborsClassifier(n_neighbors=55, weights='distance', metric='minkowski'))
])))

svm_pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), feature_set)], remainder='drop')
model_registry.append(('svm_rbf_diag', Pipeline([
    ('pre', svm_pre),
    ('clf', SVC(C=1.5, gamma='scale', kernel='rbf', probability=True, random_state=42))
])))

optional_library_status = {'xgboost': False, 'lightgbm': False}
try:
    from xgboost import XGBClassifier
    xgb_pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), feature_set)], remainder='drop')
    model_registry.append(('xgboost_strong', Pipeline([
        ('pre', xgb_pre),
        ('clf', XGBClassifier(
            n_estimators=350, max_depth=6, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
            reg_alpha=0.0, reg_lambda=1.0, objective='binary:logistic', eval_metric='auc',
            random_state=42, n_jobs=-1
        ))
    ])))
    optional_library_status['xgboost'] = True
except Exception:
    pass

try:
    from lightgbm import LGBMClassifier
    lgb_pre = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), feature_set)], remainder='drop')
    model_registry.append(('lightgbm_strong', Pipeline([
        ('pre', lgb_pre),
        ('clf', LGBMClassifier(
            n_estimators=450, num_leaves=63, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
            objective='binary', random_state=42, n_jobs=-1
        ))
    ])))
    optional_library_status['lightgbm'] = True
except Exception:
    pass

rows = []
pred_store = {}
runtime_rows = []

for name, model in model_registry:
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    t_train = time.perf_counter() - t0

    t1 = time.perf_counter()
    p_tr = model.predict_proba(X_train)[:, 1]
    p_va = model.predict_proba(X_val)[:, 1]
    p_te = model.predict_proba(X_test)[:, 1]
    t_pred = time.perf_counter() - t1

    rows.append(metric_row('J_strong_tabular', name, 'train', y_train, p_tr, train_sec=t_train, pred_sec=t_pred))
    rows.append(metric_row('J_strong_tabular', name, 'val', y_val, p_va, train_sec=t_train, pred_sec=t_pred))
    rows.append(metric_row('J_strong_tabular', name, 'test', y_test, p_te, train_sec=t_train, pred_sec=t_pred))

    runtime_rows.append({
        'model': name,
        'train_seconds': float(t_train),
        'predict_seconds_total': float(t_pred),
        'n_train': int(len(X_train)),
        'n_val': int(len(X_val)),
        'n_test': int(len(X_test))
    })

    pred_store[name] = {
        'train': p_tr,
        'val': p_va,
        'test': p_te
    }

j_metrics = pd.DataFrame(rows)
j_metrics.to_csv(TABLE_DIR / 'stage_j_model_metrics_by_split.csv', index=False)

j_gap = []
for name, g in j_metrics.groupby('model'):
    tr = float(g.loc[g['split'] == 'train', 'auroc'].iloc[0])
    va = float(g.loc[g['split'] == 'val', 'auroc'].iloc[0])
    te = float(g.loc[g['split'] == 'test', 'auroc'].iloc[0])
    ece_te = float(g.loc[g['split'] == 'test', 'ece_10bin'].iloc[0])
    j_gap.append({
        'model': name,
        'train_auroc': tr,
        'val_auroc': va,
        'test_auroc': te,
        'test_ece_10bin': ece_te,
        'train_val_gap': tr - va,
        'val_test_gap': va - te
    })
j_gap_df = pd.DataFrame(j_gap).sort_values(['val_auroc', 'test_auroc'], ascending=False)
j_gap_df.to_csv(TABLE_DIR / 'stage_j_capacity_gap_summary.csv', index=False)

runtime_df = pd.DataFrame(runtime_rows).sort_values('train_seconds')
runtime_df.to_csv(TABLE_DIR / 'stage_j_runtime_profile.csv', index=False)

# Unified H/I/J-style matrix (H placeholder if absent)
comparison_rows = []
if comparison_path.exists():
    g_comp = pd.read_csv(comparison_path)
    top_g = g_comp.sort_values('auroc', ascending=False).iloc[0].to_dict()
    comparison_rows.append({
        'notebook_stage': 'G_reference',
        'model': str(top_g['model']),
        'auroc': float(top_g['auroc']),
        'pr_auc': float(top_g['pr_auc']),
        'brier': float(top_g['brier']),
        'ece_10bin': float(top_g['ece_10bin'])
    })

i_gap_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook06_stage_i' / 'stage_i_rf_gap_summary.csv'
if i_gap_path.exists():
    i_gap = pd.read_csv(i_gap_path).sort_values('val_auroc', ascending=False).iloc[0]
    comparison_rows.append({
        'notebook_stage': 'I_selected',
        'model': str(i_gap['model']),
        'auroc': float(i_gap['test_auroc']),
        'pr_auc': np.nan,
        'brier': np.nan,
        'ece_10bin': np.nan
    })

best_j_name = j_gap_df.iloc[0]['model']
best_j_test = j_metrics[(j_metrics['model'] == best_j_name) & (j_metrics['split'] == 'test')].iloc[0]
comparison_rows.append({
    'notebook_stage': 'J_selected',
    'model': str(best_j_name),
    'auroc': float(best_j_test['auroc']),
    'pr_auc': float(best_j_test['pr_auc']),
    'brier': float(best_j_test['brier']),
    'ece_10bin': float(best_j_test['ece_10bin'])
})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(TABLE_DIR / 'stage_j_comparison_matrix.csv', index=False)

plt.figure(figsize=(10, 5))
plot_df = j_gap_df.head(12).copy()
plt.plot(plot_df['model'], plot_df['train_auroc'], marker='o', label='train')
plt.plot(plot_df['model'], plot_df['val_auroc'], marker='o', label='val')
plt.plot(plot_df['model'], plot_df['test_auroc'], marker='o', label='test')
plt.xticks(rotation=35, ha='right')
plt.ylabel('AUROC')
plt.title('Stage J Strong Tabular Sweep (Train/Val/Test)')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_j_capacity_sweep_auroc.png', dpi=140, bbox_inches='tight')
plt.close()

print('Stage J block 1 generated')
print('optional libs:', optional_library_status)
print('-', (TABLE_DIR / 'stage_j_model_metrics_by_split.csv').exists())
print('-', (TABLE_DIR / 'stage_j_capacity_gap_summary.csv').exists())
print('-', (TABLE_DIR / 'stage_j_runtime_profile.csv').exists())
print('-', (TABLE_DIR / 'stage_j_comparison_matrix.csv').exists())
print('-', (FIG_DIR / 'stage_j_capacity_sweep_auroc.png').exists())

j_gap_df.head(10)

Stage J block 1 generated
optional libs: {'xgboost': False, 'lightgbm': False}
- True
- True
- True
- True
- True


,model,train_auroc,val_auroc,test_auroc,test_ece_10bin,train_val_gap,val_test_gap
1,hist_gb_strong,1.000000,0.999999,1.000000,0.000321,7.370911e-07,-4.494533e-07
3,rf_stage_i_selected,1.000000,0.999999,0.999999,0.001860,6.519319e-07,-1.954782e-08
0,extra_trees_strong,0.999998,0.999993,0.999988,0.020377,5.405782e-06,4.376525e-06
4,svm_rbf_diag,0.999986,0.999963,0.999984,0.001191,2.246798e-05,-2.058804e-05
2,knn_diag,1.000000,0.999898,0.999862,0.003690,1.016840e-04,3.659469e-05


In [6]:
# Stage J implementation block 2: bootstrap CI + subgroup spread + promotion recommendation + manifest/proof
from sklearn.metrics import roc_auc_score

test_slice = model_df['split'] == 'test'
y_test_vec = model_df.loc[test_slice, label].astype(int).to_numpy()
test_groups = model_df.loc[test_slice, 'response_state_active'].astype(str).to_numpy()
test_patient_ids = model_df.loc[test_slice, 'patient_id'].to_numpy()

best_models = j_gap_df.head(3)['model'].tolist()
rng = np.random.default_rng(42)
B = 80

ci_rows = []
for m in best_models:
    p_test_vec = pred_store[m]['test']
    auc_samples = []
    pr_samples = []
    brier_samples = []
    ece_samples = []
    n_test = len(y_test_vec)
    for _ in range(B):
        idx = rng.integers(0, n_test, size=n_test)
        yt = y_test_vec[idx]
        pp = p_test_vec[idx]
        auc_samples.append(float(roc_auc_score(yt, pp)))
        pr_samples.append(float(average_precision_score(yt, pp)))
        brier_samples.append(float(brier_score_loss(yt, pp)))
        ece_samples.append(float(ece_score(yt, pp)))
    ci_rows.append({
        'model': m,
        'metric': 'auroc',
        'mean': float(np.mean(auc_samples)),
        'ci_lower': float(np.quantile(auc_samples, 0.025)),
        'ci_upper': float(np.quantile(auc_samples, 0.975))
    })
    ci_rows.append({
        'model': m,
        'metric': 'pr_auc',
        'mean': float(np.mean(pr_samples)),
        'ci_lower': float(np.quantile(pr_samples, 0.025)),
        'ci_upper': float(np.quantile(pr_samples, 0.975))
    })
    ci_rows.append({
        'model': m,
        'metric': 'brier',
        'mean': float(np.mean(brier_samples)),
        'ci_lower': float(np.quantile(brier_samples, 0.025)),
        'ci_upper': float(np.quantile(brier_samples, 0.975))
    })
    ci_rows.append({
        'model': m,
        'metric': 'ece_10bin',
        'mean': float(np.mean(ece_samples)),
        'ci_lower': float(np.quantile(ece_samples, 0.025)),
        'ci_upper': float(np.quantile(ece_samples, 0.975))
    })

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(TABLE_DIR / 'stage_j_bootstrap_ci.csv', index=False)

subgroup_rows = []
for m in best_models:
    p_test_vec = pred_store[m]['test']
    overall_auc = float(roc_auc_score(y_test_vec, p_test_vec))
    for grp in sorted(pd.unique(test_groups)):
        mask = test_groups == grp
        if mask.sum() < 50 or len(np.unique(y_test_vec[mask])) < 2:
            continue
        auc = float(roc_auc_score(y_test_vec[mask], p_test_vec[mask]))
        pr = float(average_precision_score(y_test_vec[mask], p_test_vec[mask]))
        brier = float(brier_score_loss(y_test_vec[mask], p_test_vec[mask]))
        subgroup_rows.append({
            'model': m,
            'group': grp,
            'n': int(mask.sum()),
            'auroc': auc,
            'pr_auc': pr,
            'brier': brier,
            'auroc_delta_vs_overall': auc - overall_auc
        })

subgroup_df = pd.DataFrame(subgroup_rows)
subgroup_df.to_csv(TABLE_DIR / 'stage_j_subgroup_delta.csv', index=False)

spread_rows = []
for m, g in subgroup_df.groupby('model'):
    spread_rows.append({
        'model': m,
        'subgroup_auroc_spread': float(g['auroc'].max() - g['auroc'].min()),
        'subgroup_brier_spread': float(g['brier'].max() - g['brier'].min())
    })
spread_df = pd.DataFrame(spread_rows)
spread_df.to_csv(TABLE_DIR / 'stage_j_subgroup_spread_summary.csv', index=False)

# Promotion: maximize val AUROC, then prefer smaller calibration & subgroup risk spreads
candidate = j_gap_df.merge(
    spread_df, on='model', how='left'
).fillna({'subgroup_auroc_spread': 0.0, 'subgroup_brier_spread': 0.0})
candidate['safety_penalty'] = (
    candidate['test_ece_10bin'].abs()
    + candidate['train_val_gap'].abs()
    + candidate['val_test_gap'].abs()
    + candidate['subgroup_auroc_spread'].abs()
    + candidate['subgroup_brier_spread'].abs() * 10.0
)
selected_j = candidate.sort_values(['val_auroc', 'safety_penalty'], ascending=[False, True]).iloc[0]

summary_lines = [
    'Stage J Strong Tabular Promotion Summary',
    f"optional_xgboost_available: {optional_library_status['xgboost']}",
    f"optional_lightgbm_available: {optional_library_status['lightgbm']}",
    f"n_models_evaluated: {j_metrics['model'].nunique()}",
    f"selected_model: {selected_j['model']}",
    f"selected_val_auroc: {selected_j['val_auroc']:.6f}",
    f"selected_test_auroc: {selected_j['test_auroc']:.6f}",
    f"selected_test_ece: {selected_j['test_ece_10bin']:.6f}",
    f"selected_train_val_gap: {selected_j['train_val_gap']:.6e}",
    f"selected_val_test_gap: {selected_j['val_test_gap']:.6e}",
    f"selected_subgroup_auroc_spread: {selected_j['subgroup_auroc_spread']:.6f}",
    f"selected_subgroup_brier_spread: {selected_j['subgroup_brier_spread']:.6f}"
]
with open(REPORT_DIR / 'stage_j_promotion_recommendation.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))

manifest_j = {
    'phase': 'J',
    'notebook': '07_stage_j_strong_tabular_models.ipynb',
    'inputs': [
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet',
        'Data/metadata/phase_i_manifest.json',
        'Results/reports/notebook06_stage_i/stage_i_checklist_proof.json'
    ],
    'outputs_tables': [
        'Results/tables/notebook07_stage_j/stage_j_model_metrics_by_split.csv',
        'Results/tables/notebook07_stage_j/stage_j_capacity_gap_summary.csv',
        'Results/tables/notebook07_stage_j/stage_j_runtime_profile.csv',
        'Results/tables/notebook07_stage_j/stage_j_comparison_matrix.csv',
        'Results/tables/notebook07_stage_j/stage_j_bootstrap_ci.csv',
        'Results/tables/notebook07_stage_j/stage_j_subgroup_delta.csv',
        'Results/tables/notebook07_stage_j/stage_j_subgroup_spread_summary.csv'
    ],
    'outputs_figures': [
        'Results/figures/notebook07_stage_j/stage_j_capacity_sweep_auroc.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook07_stage_j/stage_j_promotion_recommendation.txt'
    ],
    'selected_model': str(selected_j['model']),
    'optional_library_status': optional_library_status
}
with open(META_DIR / 'phase_j_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_j, f, indent=4)

proof_j = {
    'metrics_generated': (TABLE_DIR / 'stage_j_model_metrics_by_split.csv').exists(),
    'comparison_generated': (TABLE_DIR / 'stage_j_comparison_matrix.csv').exists(),
    'bootstrap_ci_generated': (TABLE_DIR / 'stage_j_bootstrap_ci.csv').exists(),
    'subgroup_delta_generated': (TABLE_DIR / 'stage_j_subgroup_delta.csv').exists(),
    'recommendation_generated': (REPORT_DIR / 'stage_j_promotion_recommendation.txt').exists(),
    'manifest_generated': (META_DIR / 'phase_j_manifest.json').exists()
}
with open(REPORT_DIR / 'stage_j_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof_j, 'selected': selected_j.to_dict()}, f, indent=4)

print('Stage J block 2 generated')
for k, v in proof_j.items():
    print('-', k, ':', v)

candidate.sort_values(['val_auroc', 'safety_penalty'], ascending=[False, True]).head(10)

Stage J block 2 generated
- metrics_generated : True
- comparison_generated : True
- bootstrap_ci_generated : True
- subgroup_delta_generated : True
- recommendation_generated : True
- manifest_generated : True


,model,train_auroc,val_auroc,test_auroc,test_ece_10bin,train_val_gap,val_test_gap,subgroup_auroc_spread,subgroup_brier_spread,safety_penalty
0,hist_gb_strong,1.000000,0.999999,1.000000,0.000321,7.370911e-07,-4.494533e-07,0.000003,0.000689,0.007217
1,rf_stage_i_selected,1.000000,0.999999,0.999999,0.001860,6.519319e-07,-1.954782e-08,0.000004,0.000551,0.007377
2,extra_trees_strong,0.999998,0.999993,0.999988,0.020377,5.405782e-06,4.376525e-06,0.000034,0.001164,0.032064
3,svm_rbf_diag,0.999986,0.999963,0.999984,0.001191,2.246798e-05,-2.058804e-05,0.000000,0.000000,0.001234
4,knn_diag,1.000000,0.999898,0.999862,0.003690,1.016840e-04,3.659469e-05,0.000000,0.000000,0.003828


## Execution Notes

Use this notebook as the Stage J execution surface. Keep frozen split policy and metric helpers aligned with Notebook 06.

Expected Stage J key artifacts:
- `Results/tables/notebook07_stage_j/stage_j_comparison_matrix.csv`
- `Results/tables/notebook07_stage_j/stage_j_bootstrap_ci.csv`
- `Results/tables/notebook07_stage_j/stage_j_subgroup_delta.csv`
- `Results/reports/notebook07_stage_j/stage_j_promotion_recommendation.txt`
- `Data/metadata/phase_j_manifest.json`

In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook07'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)